1: Đọc dữ liệu gốc cục bộ

In [1]:
import os
import re
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Tự động tìm dữ liệu cục bộ; nếu chạy trên Colab chưa có file thì tải từ Kaggle
cwd = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in [cwd, *cwd.parents] if (p / "data" / "raw").is_dir()),
    cwd,
)
csv_path = PROJECT_DIR / "data" / "raw" / "combined_data.csv"

if not csv_path.is_file():
    import kagglehub
    dl_path = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")
    csv_path = Path(dl_path) / "combined_data.csv"

df = pd.read_csv(csv_path)
print(f"Đường dẫn dữ liệu: {csv_path}")
print(f"Kích thước dữ liệu gốc: {df.shape[0]:,} dòng, {df.shape[1]} cột")


Đường dẫn dữ liệu: D:\spam-email-classifier\data\raw\combined_data.csv
Kích thước dữ liệu gốc: 83,448 dòng, 2 cột


2: Làm sạch dữ liệu

In [2]:
# 1. Xóa giá trị thiếu (nếu có)
df_clean = df.dropna(subset=["text", "label"]).copy()

# 2. Chuẩn hóa văn bản:
#    - Đưa dấu nháy/gạch nối Unicode về ASCII chuẩn, xóa ký tự lỗi \ufffd
#    - Loại bỏ các ký tự non-ASCII / lỗi font mojibake (ví dụ chuỗi Latin-1 bị lỗi mã hóa)
#    - Chuyển chữ thường và gom nhiều khoảng trắng thành 1
PUNCT_TRANS = str.maketrans({
    "\u2018": "'", "\u2019": "'",
    "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-",
    "\ufffd": " ",
})

def clean_text(text):
    text = str(text).translate(PUNCT_TRANS).lower()
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_clean["text"] = df_clean["text"].apply(clean_text)

# 3. Loại bỏ các dòng rỗng hoặc không còn ký tự chữ/số sau khi làm sạch
valid_mask = df_clean["text"].str.contains(r"[a-z0-9]", regex=True)
print(f"Số dòng rỗng / chỉ chứa ký tự lỗi bị loại bỏ: {(~valid_mask).sum():,}")
df_clean = df_clean[valid_mask].copy()

# 4. Loại bỏ các dòng trùng nội dung có nhãn mâu thuẫn, sau đó khử trùng lặp nội dung
conflicts = df_clean.groupby("text")["label"].transform("nunique").gt(1)
print(f"Số dòng trùng nội dung nhưng khác nhãn bị loại bỏ: {int(conflicts.sum()):,}")
df_clean = df_clean[~conflicts]

before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Số dòng trùng nội dung cùng nhãn bị loại bỏ: {before_dedup - len(df_clean):,}")
print(f"Kích thước sau khi làm sạch: {df_clean.shape[0]:,} dòng")


Số dòng rỗng / chỉ chứa ký tự lỗi bị loại bỏ: 13
Số dòng trùng nội dung nhưng khác nhãn bị loại bỏ: 6
Số dòng trùng nội dung cùng nhãn bị loại bỏ: 468
Kích thước sau khi làm sạch: 82,961 dòng


3: Chia tập Train (80%) và Test (20%)

In [3]:
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['label']
)

print(f"Tập Train: {train_df.shape[0]:,} dòng")
print(f"Tập Test : {test_df.shape[0]:,} dòng")

print("\nTỷ lệ nhãn tập Train (%):")
print((train_df['label'].value_counts(normalize=True) * 100).round(2))

print("\nTỷ lệ nhãn tập Test (%):")
print((test_df['label'].value_counts(normalize=True) * 100).round(2))

Tập Train: 66,368 dòng
Tập Test : 16,593 dòng

Tỷ lệ nhãn tập Train (%):
label
1    52.73
0    47.27
Name: proportion, dtype: float64

Tỷ lệ nhãn tập Test (%):
label
1    52.73
0    47.27
Name: proportion, dtype: float64


4: Lưu tập train và test vào data/processed

In [4]:
output_dir = PROJECT_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "train.csv"
test_path = output_dir / "test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Đã lưu tập train tại: {train_path}")
print(f"Đã lưu tập test  tại: {test_path}")

# Nếu đang chạy trên Google Colab thì nén và tải về
try:
    import shutil
    from google.colab import files
    shutil.make_archive("processed_data", "zip", output_dir)
    files.download("processed_data.zip")
except ImportError:
    pass


Đã lưu tập train tại: D:\spam-email-classifier\data\processed\train.csv
Đã lưu tập test  tại: D:\spam-email-classifier\data\processed\test.csv
